# Entrega final - Agente Connect-4 con rollouts heuristicos

Este notebook presenta la entrega final del agente para Connect-4. El objetivo es demostrar, con experimentos empiricos, que el desempeno no depende de una sola etiqueta del agente, sino de variables como la cantidad de trials, el tipo de rollout, el oponente y el orden de juego.

**Agente final:** `Agente con rollouts heuristicos`, una politica de Trial-Based Online Policy Improvement que usa una heuristica de Connect-4 durante las simulaciones.  
**Baseline principal:** `Agente con rollouts aleatorios`, la misma tecnica de mejora online, pero con simulaciones aleatorias.  
**Baseline minimo:** `Random`, jugador que elige una columna legal al azar.


## 1. Requisitos de la rubrica cubiertos

- **Codigo del agente:** `groups/Group S/policy.py` y `groups/Group S/Euristic.py`.
- **Datos experimentales:** `resultados_experimentos.csv`, con una fila por partida y perspectiva focal.
- **Notebook de estudio:** este archivo `entrega.ipynb` carga los datos, resume metricas y muestra las graficas de validacion.
- **Variables analizadas:** version del agente, cantidad de trials, oponente y color/orden de juego.
- **Oponentes minimos de la rubrica:** jugador aleatorio (`Random`) y self-play contra una copia del agente.
- **Comparacion entre versiones:** agente con rollouts heuristicos contra agente con rollouts aleatorios, para medir si la heuristica mejora la version base.
- **Conclusiones empiricas:** cada conclusion se apoya en tablas o graficas generadas desde el CSV.


## 2. Idea principal del agente

Ambos agentes usan la misma familia de metodo: **Trial-Based Online Policy Improvement**. En cada turno se revisan las acciones legales, se toma cada accion candidata, se simulan partidas completas desde el estado resultante y se estima el valor promedio de esa accion. Despues, el agente juega la columna con mejor promedio.

La diferencia importante no esta en la estructura externa del algoritmo, sino en la politica usada dentro de cada simulacion:

- **Agente con rollouts aleatorios:** durante los trials elige movimientos legales al azar. Es rapido y da diversidad, pero sus estimaciones son ruidosas porque muchas simulaciones no representan buen juego.
- **Agente con rollouts heuristicos:** durante los trials usa conocimiento del juego: ganar si puede, bloquear victorias inmediatas del rival, preferir columnas centrales, construir amenazas y evitar regalar jugadas ganadoras.

Por eso el experimento compara dos cosas separadas. Primero, si la mejora de politica online funciona contra `Random`. Segundo, si cambiar el rollout aleatorio por uno heuristico mejora la calidad de decision cuando ambos agentes usan el mismo enfoque de simulaciones. La hipotesis es que la heuristica necesita menos trials para tomar buenas decisiones, aunque cada trial sea mas costoso computacionalmente.


## 3. Carga de datos y resumen reproducible

El CSV ya contiene los resultados de las simulaciones. Para no repetir una ejecucion de varias horas, el notebook carga `resultados_experimentos.csv` y genera metricas desde ese archivo.


In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd()
RESULTS_PATH = PROJECT_ROOT / 'resultados_experimentos.csv'
FIG_DIR = PROJECT_ROOT / 'figuras_entrega'

df = pd.read_csv(RESULTS_PATH)
for col in ['trials', 'opponent_trials', 'configuration_trials']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print('Filas:', len(df))
print('Partidas reales:', df['match_id'].nunique())
print('Agentes:', sorted(df['focal_agent'].dropna().unique()))
print('Trials disponibles:', sorted(df['configuration_trials'].dropna().astype(int).unique()))

df.head()


### Resumen del dataset

- Filas en `resultados_experimentos.csv`: **4100**.
- Partidas reales: **2050**.
- Trials evaluados: **1, 5, 10, 25, 50, 100**.
- El agente con rollouts aleatorios tiene mas partidas porque es mucho mas barato de ejecutar.
- El agente con rollouts heuristicos tiene menos partidas por configuracion porque cada simulacion evalua reglas del juego.


In [ ]:
# Opcional: regenerar las figuras desde resultados_experimentos.csv.
# Esta celda esta desactivada por defecto porque las imagenes ya fueron exportadas.
# Para regenerarlas, cambia REGENERAR_FIGURAS = True y ejecuta la celda.
REGENERAR_FIGURAS = False

if REGENERAR_FIGURAS:
    import runpy
    runpy.run_path('generar_graficas_entrega.py')


## 4. Protocolo experimental

Se evaluaron estos escenarios:

1. Agente con rollouts aleatorios vs `Random`, jugando primero y segundo.
2. Agente con rollouts heuristicos vs `Random`, jugando primero y segundo.
3. Agente con rollouts heuristicos vs agente con rollouts aleatorios, jugando primero y segundo.
4. Self-play del agente con rollouts aleatorios.
5. Self-play del agente con rollouts heuristicos.

Metricas principales:

- `win_rate`, `draw_rate`, `loss_rate`.
- `mean_score`, donde victoria = 1, empate = 0, derrota = -1.
- `mean_decision_time`, tiempo promedio que tarda el agente focal por jugada.
- `num_moves`, duracion de la partida.


## 5. Tabla: desempeno contra Random

Esta tabla resume el requisito minimo del reto: el agente debe jugar de forma robusta contra un jugador aleatorio.


| Version              | Trials | Partidas | Victorias | Empates | Derrotas | Win rate | Loss rate | Tiempo/jugada |
| -------------------- | ------ | -------- | --------- | ------- | -------- | -------- | --------- | ------------- |
| Rollouts aleatorios  | 1      | 200      | 141       | 1       | 58       | 70.5%    | 29.0%     | 0.043s        |
| Rollouts aleatorios  | 5      | 200      | 185       | 0       | 15       | 92.5%    | 7.5%      | 0.063s        |
| Rollouts aleatorios  | 10     | 200      | 194       | 0       | 6        | 97.0%    | 3.0%      | 0.118s        |
| Rollouts aleatorios  | 25     | 200      | 200       | 0       | 0        | 100.0%   | 0.0%      | 0.186s        |
| Rollouts aleatorios  | 50     | 200      | 200       | 0       | 0        | 100.0%   | 0.0%      | 0.539s        |
| Rollouts aleatorios  | 100    | 200      | 200       | 0       | 0        | 100.0%   | 0.0%      | 0.475s        |
| Rollouts heuristicos | 1      | 20       | 20        | 0       | 0        | 100.0%   | 0.0%      | 0.061s        |
| Rollouts heuristicos | 5      | 20       | 20        | 0       | 0        | 100.0%   | 0.0%      | 0.271s        |
| Rollouts heuristicos | 10     | 20       | 20        | 0       | 0        | 100.0%   | 0.0%      | 0.591s        |
| Rollouts heuristicos | 25     | 20       | 20        | 0       | 0        | 100.0%   | 0.0%      | 1.445s        |
| Rollouts heuristicos | 50     | 20       | 20        | 0       | 0        | 100.0%   | 0.0%      | 3.130s        |


## 6. Grafica 1: Win / Draw / Loss vs trials contra Random

La grafica muestra, para cada version y color, la proporcion de victorias, empates y derrotas. El eje X indica el numero de trials y `n`, la cantidad de partidas de esa configuracion. El titulo se simplifico y la leyenda se movio al pie para que no se sobreponga con los paneles.

![Grafica 1](figuras_entrega/01_wdl_vs_trials_random.png)

**Lectura principal:** el agente con rollouts heuristicos no pierde contra `Random` en ningun valor de trials evaluado. El agente con rollouts aleatorios tambien se estabiliza, pero necesita mas trials para eliminar derrotas.


## 7. Grafica 2: Win rate vs trials contra Random

![Grafica 2](figuras_entrega/02_win_rate_vs_trials_random.png)

**Lectura principal:** la version heuristica alcanza 100% de win rate desde 1 trial. La version con rollouts aleatorios mejora gradualmente y alcanza 100% desde 25 trials.


## 8. Grafica 3: Loss rate vs trials contra Random

![Grafica 3](figuras_entrega/03_loss_rate_vs_trials_random.png)

**Lectura principal:** `Agente con rollouts heuristicos` mantiene 0% de derrotas contra `Random`. `Agente con rollouts aleatorios` tiene derrotas con pocos trials, especialmente cuando juega segundo, pero las elimina al aumentar el presupuesto.


## 9. Metricas: heuristico vs rollouts aleatorios

Esta es la comparacion mas importante para demostrar que la heuristica aporta algo frente a la version base. En este bloque, ambos jugadores usan Trial-Based Online Policy Improvement; lo que cambia es si el rollout interno es aleatorio o guiado por heuristica.


| Trials | Partidas | Victorias heuristico | Empates | Derrotas heuristico | Win rate heuristico | Score heuristico | Tiempo heuristico/jugada |
| ------ | -------- | -------------------- | ------- | ------------------- | ------------------- | ---------------- | ------------------------ |
| 1      | 20       | 20                   | 0       | 0                   | 100.0%              | 1                | 0.057s                   |
| 5      | 20       | 20                   | 0       | 0                   | 100.0%              | 1                | 0.277s                   |
| 10     | 20       | 18                   | 1       | 1                   | 90.0%               | 0.85             | 0.519s                   |
| 25     | 20       | 15                   | 0       | 5                   | 75.0%               | 0.5              | 1.096s                   |
| 50     | 20       | 18                   | 0       | 2                   | 90.0%               | 0.8              | 1.959s                   |


## 10. Grafica 4: Comparacion directa entre versiones

![Grafica 4](figuras_entrega/04_head_to_head_h_vs_r.png)

**Lectura principal:** el agente con rollouts heuristicos supera al agente con rollouts aleatorios en la mayoria de configuraciones, tanto jugando primero como segundo. La variacion en trials 25 y 50 es esperable porque la version heuristica tiene menos partidas por punto experimental.


## 11. Grafica 9: Metricas explicitas de heuristico vs rollouts aleatorios

![Grafica 9](figuras_entrega/09_metricas_heuristico_vs_aleatorio.png)

**Lectura principal:** la version heuristica obtiene score positivo contra la version con rollouts aleatorios en todos los trials agregados. Esto respalda que la mejora no solo sirve contra `Random`, sino tambien frente a una version base del mismo enfoque.


## 12. Grafica 5: Self-play

![Grafica 5](figuras_entrega/05_self_play_score.png)

**Lectura principal:** en self-play no se espera win rate alto porque ambos lados usan la misma politica. La grafica sirve para observar ventaja del primer jugador, estabilidad y empates.


### Tabla: self-play


| Version              | Trials | Partidas | Score jugador 1 | Draw rate | Movimientos |
| -------------------- | ------ | -------- | --------------- | --------- | ----------- |
| Rollouts aleatorios  | 1      | 100      | 0.2             | 2.0%      | 20.39       |
| Rollouts aleatorios  | 5      | 100      | 0.12            | 0.0%      | 16.46       |
| Rollouts aleatorios  | 10     | 100      | 0.23            | 1.0%      | 16.51       |
| Rollouts aleatorios  | 25     | 100      | 0.1             | 0.0%      | 18.95       |
| Rollouts aleatorios  | 50     | 100      | 0.19            | 1.0%      | 22.83       |
| Rollouts aleatorios  | 100    | 100      | 0.04            | 4.0%      | 26.64       |
| Rollouts heuristicos | 1      | 10       | 0.2             | 20.0%     | 37.1        |
| Rollouts heuristicos | 5      | 10       | 0.6             | 40.0%     | 35          |
| Rollouts heuristicos | 10     | 10       | 0               | 20.0%     | 34.6        |
| Rollouts heuristicos | 25     | 10       | -0.5            | 30.0%     | 35.7        |
| Rollouts heuristicos | 50     | 10       | 0.3             | 30.0%     | 35.9        |


## 13. Grafica 6: Heatmap de enfrentamientos

![Grafica 6](figuras_entrega/06_heatmap_score_trials_50.png)

**Lectura principal:** el heatmap resume el score promedio del jugador que empieza. Trials=50 se usa porque es el maximo comun entre `Agente con rollouts heuristicos` y `Agente con rollouts aleatorios` en los datos finales.


## 14. Grafica 7: Tiempo promedio por jugada vs trials

![Grafica 7](figuras_entrega/07_time_per_move_vs_trials.png)

**Lectura principal:** `Agente con rollouts heuristicos` es mas costoso por decision. Esto ocurre porque cada rollout heuristico evalua movimientos prometedores en vez de elegir columnas al azar. El beneficio es mejor rendimiento con pocos trials; el costo es mayor tiempo por jugada.


## 15. Grafica 8: Rendimiento vs costo computacional

![Grafica 8](figuras_entrega/08_performance_vs_cost.png)

**Lectura principal:** `Agente con rollouts heuristicos` domina en rendimiento con pocos trials, pero no siempre en eficiencia temporal. `Agente con rollouts aleatorios` con 25 trials es una configuracion eficiente si se prioriza velocidad, mientras que `Agente con rollouts heuristicos` es mejor si se prioriza calidad de decision con bajo numero de trials.


## 16. Conclusiones

1. **La heuristica mejora la calidad de los rollouts.** El agente con rollouts heuristicos alcanza 100% de victorias contra `Random` desde 1 trial, mientras el agente con rollouts aleatorios necesita mas trials para estabilizarse.
2. **La mejora tambien aparece en la comparacion directa.** La version heuristica vence con frecuencia a la version con rollouts aleatorios, lo que indica que la heuristica aporta mas que solo explotar debilidades del jugador aleatorio.
3. **El costo computacional es la debilidad principal.** El agente heuristico tarda mas por jugada porque cada paso simulado requiere evaluacion heuristica.
4. **El orden de juego importa.** Por eso las graficas separan primer y segundo jugador, como pide el analisis por variables experimentales.
5. **Self-play aporta estabilidad al analisis.** Permite observar si el agente se comporta razonablemente contra una copia de si mismo y si hay ventaja del primer jugador.


## 17. Debilidades y mejoras futuras

- **Debilidad:** el agente con rollouts heuristicos puede ser lento con muchos trials.  
  **Mejora:** usar asignacion adaptativa de trials, deteniendo temprano acciones claramente inferiores.

- **Debilidad:** algunas configuraciones de comparacion directa tienen alta varianza por tener pocas partidas.  
  **Mejora:** aumentar partidas solo en los puntos criticos, por ejemplo trials 25 y 50.

- **Debilidad:** el rollout heuristico puede perder diversidad si siempre favorece las mismas acciones.  
  **Mejora:** mantener muestreo estocastico entre las mejores acciones y ajustar la temperatura de seleccion.

- **Debilidad:** el agente puede gastar tiempo en estados ya tacticamente claros.  
  **Mejora:** detectar victorias, bloqueos y amenazas forzadas antes de lanzar todos los trials.


## 18. Figuras recomendadas para una presentacion de 3 minutos

Para una sustentacion corta usaria estas cuatro:

1. Grafica 1: Win / Draw / Loss contra Random.
2. Grafica 2: Win rate vs trials, rollouts heuristicos vs rollouts aleatorios.
3. Grafica 9: Metricas explicitas de heuristico vs rollouts aleatorios.
4. Grafica 8: Rendimiento vs costo computacional.


In [ ]:
# Celda opcional para revisar que todas las figuras esten disponibles
from pathlib import Path
for path in sorted((Path.cwd() / 'figuras_entrega').glob('*.png')):
    print(path.name)
